In [5]:
import numpy as np
import torch 

train_data = np.memmap("./sample_dataset/train.bin",dtype=np.uint16,mode="r")
val_data = np.memmap("./sample_dataset/val.bin",dtype=np.uint16,mode="r")

def get_batches(split="train",batch_size=4,block_size=64,device="cpu"):

    # select the data source
    d = train_data if split=="train" else val_data

    # finding the index
    ix = torch.randint(0,len(d)-block_size,(batch_size,))

    # slicing x and y
    x_list = [torch.from_numpy((d[i:i+block_size]).astype(np.int64)) for i in ix]
    y_list = [torch.from_numpy((d[i+1:i+1+block_size]).astype(np.int64)) for i in ix]


    # stack the individual 1D to the 2D (B,T)
    x = torch.stack(x_list)
    y = torch.stack(y_list)

    if device != "cpu":
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
        
    return x, y

h = get_batches(split="train",batch_size=4,block_size=64,device="cpu")
h

    

(tensor([[ 4587,   198,   220,   220, 13355,    67,   260,    72,  2671,    64,
            549,   641,  7527,   353, 24328,    25,   773,    68, 39683,   318,
             83,   257,   794,   287,  4587,  7491,   301,   695,  2150,   307,
             72,   198,   220,   220, 12182, 47223,   299,   333, 18042,  1976,
          42990,   327,   641, 29578,   357, 36369,   647,  3318, 41167,     8,
           4656,  2297,    68,  8183,   628,   220,   220,   685, 41133, 39683,
          11295,   807,    25,  6733],
         [35323,  7840,   966,   286,   262, 15489,   314,  2497,   257,  2347,
            286,  2330,  5563,    11,   543,   198, 44092,   423,   587,  2035,
          29804,   393, 14891,   526,   628,   198,  1537, 14526,   373,   407,
            284,  9240,  1363,  1231,  1194,  3735,  1483, 12077,   465,   198,
          10608,    13,  1550,  3267,  1511,   400,    11,   618,  2045,   329,
            257, 12538,   287,   257,  7850,    11,   530,   286,   198,  1169,
 

# Data Loader: Efficient Batch Streaming & Memory Mapping

Since we have saved the `train.bin` and `val.bin` binary files on disk, we use NumPy's `np.memmap` method to stream them directly from disk in read mode as unsigned 16-bit integers (`np.uint16`).

We create a `get_batches` method that splits the dataset into `train` and `val` sets depending on our training needs.

### 1. Random Batch Indexing & Preventing Overflow

When choosing random starting indices, `torch.randint(0, len(d) - block_size, (batch_size,))` picks starting positions between `0` and `len(d) - block_size - 1`.

### Why do we need to subtract `block_size`?

To avoid index overflow at the end of the file! If we start a batch at an index near the very end of the dataset, there won't be enough remaining tokens to fill a full sequence of length `block_size`.

For example, if a dataset has 200 tokens and `block_size = 6`:
- We start between `0` and `200 - 6 - 1 = 193`.
- The maximum starting index for $X$ is `193` (covering tokens `193` to `198`).
- Target $Y$ is shifted by $+1$ (covering tokens `194` to `199`), which cleanly hits the end of the array without index out-of-bounds overflow.

### 2. Converting Slices to 64-bit PyTorch Tensors

Once we get our random starting indices `ix`, we slice the data step-by-step:
- For input $X$: slice `d[i : i + block_size]`
- For target $Y$: slice `d[i + 1 : i + 1 + block_size]`

We convert the NumPy slices to PyTorch tensors with `int64` dtype (long integers). PyTorch expects 64-bit integers for embedding lookups because its underlying C++ backend requires `int64` tensor indices. If we pass 16-bit integers directly, PyTorch will throw an index error.

### 3. Stacking 1D Slices into 2D Batches $(B, T)$

The individual slices are 1D tensors of shape $(T,)$. We use `torch.stack()` to combine these individual 1D sequences into a 2D batch tensor of shape $(B, T)$.

### 4. GPU Memory Pinning & Async Data Transfer

When transferring data from CPU to GPU, we optimize performance using memory pinning and non-blocking copies:

#### Standard Host-to-Device Copy (`.to(device)`)
CPU RAM and GPU VRAM are physically separate hardware chips connected via PCIe lanes. Calling `.to("cuda")` triggers a DMA (Direct Memory Access) transfer pushing the tensor across the motherboard bus into GPU memory.

#### Pinned Page-Locked Memory (`.pin_memory()`)
Normally, the OS kernel pages regular RAM in and out of disk swap space. Because regular RAM addresses can shift, the GPU cannot safely copy from it directly. The CPU has to first copy the tensor to a "pinned" (page-locked) buffer, then copy it to the GPU (two copies).

Calling `.pin_memory()` locks that chunk of CPU RAM directly, allowing the GPU to read it in one direct jump via PCIe.

#### Asynchronous Overlap (`non_blocking=True`)
By default, `.to("cuda")` pauses Python execution until every byte reaches the GPU.

Setting `non_blocking=True` tells Python: *"Start transferring this batch to the GPU in the background but don't freeze the CPU thread. Let Python keep running."*

This allows the CPU to fetch and prepare the next data batch while the GPU is actively computing the current forward/backward pass.
